# Stress Monitoring — EEG + GSR Binary Classifier V2

**Goal:** improve genuine unseen-subject performance for **NORMAL vs HIGH stress**.

This V2 keeps the strict subject-independent train/validation/test split. It does **not** alter test labels, duplicate test subjects into training, or select a threshold using the test set. The notebook includes stronger EEG/GSR representations, reduced-overfitting CNN branches, balanced training, focal loss, conservative physiological-image augmentation, gated multimodal fusion, validation threshold tuning, and a 3-seed ensemble.

> **Important:** 90% is a target, not a guaranteed result. The previous model reached 59.88% strict unseen-subject accuracy with ROC-AUC 0.6493, so no honest notebook can promise 90% without seeing whether the underlying dataset contains enough separable signal.


In [ ]:
# ============================================================
# 1. SETUP + LOAD DATASET FROM GOOGLE DRIVE
# ============================================================
# This cell is made to work with either:
# A) stress_dataset_colab.zip inside My Drive, OR
# B) a shared Dataset/stress_dataset folder added as a shortcut
#    to My Drive.
#
# IMPORTANT:
# In your father's Google Drive, if "Dataset" is under
# "Shared with me", use:
# Right-click Dataset -> Organize -> Add shortcut -> My Drive.
# Then run this cell. You do NOT need to change the rest of the notebook.

import os, zipfile, shutil, gc, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
DATASET_DIR = Path("/content/stress_dataset")
MANIFEST_PATH = DATASET_DIR / "window_manifest.csv"
WINDOW_DIR = DATASET_DIR / "windows"

ZIP_NAME = "stress_dataset_colab.zip"

# ---------- Find dataset automatically ----------
zip_matches = list(DRIVE_ROOT.rglob(ZIP_NAME))

folder_candidates = []
for name in ["Dataset", "stress_dataset", "stress_dataset_colab"]:
    folder_candidates += [p for p in DRIVE_ROOT.rglob(name) if p.is_dir()]

source_folder = None

if zip_matches:
    DRIVE_ZIP_PATH = zip_matches[0]
    print("Found ZIP:", DRIVE_ZIP_PATH)

    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f"Corrupt ZIP entry: {bad}")
        z.extractall(DATASET_DIR)

else:
    # Look for a folder containing the expected dataset structure.
    for candidate in folder_candidates:
        if (candidate / "window_manifest.csv").exists() and (candidate / "windows").is_dir():
            source_folder = candidate
            break

    if source_folder is None:
        raise FileNotFoundError(
            "\nDataset not found.\n"
            "Either:\n"
            "1) put stress_dataset_colab.zip somewhere inside My Drive, OR\n"
            "2) add the shared Dataset folder as a shortcut to My Drive.\n"
            "The folder must contain window_manifest.csv and a windows folder."
        )

    print("Found dataset folder:", source_folder)

    # Copy from Drive to Colab local storage for faster processing.
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(source_folder, DATASET_DIR)

assert MANIFEST_PATH.exists(), "window_manifest.csv missing"
assert WINDOW_DIR.exists(), "windows directory missing"

base_manifest = pd.read_csv(MANIFEST_PATH)

print("\nDataset ready")
print("Dataset location used:", DATASET_DIR)
print("Subjects:", base_manifest.subject.nunique())
print("NPZ files:", len(list(WINDOW_DIR.glob("*.npz"))))
print("Original windows:", len(base_manifest))
print("\nYou do NOT need to change any other dataset path in this notebook.")


In [ ]:
# ============================================================
# 2. V2 PHYSIOLOGY-PRESERVING IMAGE GENERATION
# ============================================================
# Changes from V1:
# - EEG: 5 bands (delta/theta/alpha/beta/gamma) represented as
#   three robust composite channels using RELATIVE band power.
# - GSR: raw + tonic + phasic/derivative representation.
# - Train-only GSR scaling prevents split leakage.
# - 30-sec segments with a moderate 15-sec stride reduce redundancy.
# ============================================================

from scipy.signal import spectrogram, savgol_filter
from PIL import Image, ImageDraw

OUT_ROOT = Path("/content/stress_image_dataset_binary_v2")
EEG_DIR, GSR_DIR = OUT_ROOT/"eeg", OUT_ROOT/"gsr"
MANIFEST_OUT = OUT_ROOT/"image_manifest_binary_v2.csv"
EEG_DIR.mkdir(parents=True, exist_ok=True); GSR_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
EEG_FS = 128
SEGMENT_WINDOWS = 6       # 6 x 5 s = 30 s
SEGMENT_STRIDE = 3        # 15 s step; less redundant than old 10 s step


def decode_phase(x): return x.decode() if isinstance(x, bytes) else str(x)

def smooth_gsr(x):
    x = np.asarray(x, dtype=np.float32)
    return savgol_filter(x, 11, 2, mode="interp").astype(np.float32) if len(x) >= 11 else x.copy()

def robust_uint8(x, lo=None, hi=None):
    x = np.nan_to_num(np.asarray(x, dtype=np.float32), nan=0, posinf=0, neginf=0)
    if lo is None: lo = np.percentile(x, 2)
    if hi is None: hi = np.percentile(x, 98)
    if hi <= lo: hi = lo + 1e-6
    return (np.clip((x-lo)/(hi-lo),0,1)*255).astype(np.uint8)

def eeg_v2_image(eeg_30s):
    # Five physiologically meaningful bands.
    bands = [(0.5,4),(4,8),(8,13),(13,30),(30,45)]
    chans=[]
    for ch in range(eeg_30s.shape[0]):
        f,t,sxx=spectrogram(eeg_30s[ch],fs=EEG_FS,nperseg=256,noverlap=192,nfft=256,scaling="density",mode="psd")
        powers=[]
        total=np.mean(sxx[(f>=0.5)&(f<45)],axis=0)+1e-12
        for lo,hi in bands:
            m=(f>=lo)&(f<hi)
            p=np.mean(sxx[m],axis=0)+1e-12
            # Relative power is more robust to subject-to-subject amplitude differences.
            powers.append(np.log10(p/total))
        chans.append(np.stack(powers,axis=-1))
    feat=np.stack(chans,axis=0)
    # Compress 5 bands into 3 channels while retaining all bands:
    # R=delta+theta, G=alpha+theta+beta, B=beta+gamma.
    rgb=np.zeros((feat.shape[0],feat.shape[1],3),dtype=np.uint8)
    rgb[...,0]=robust_uint8(0.55*feat[...,0]+0.45*feat[...,1])
    rgb[...,1]=robust_uint8(0.25*feat[...,1]+0.50*feat[...,2]+0.25*feat[...,3])
    rgb[...,2]=robust_uint8(0.55*feat[...,3]+0.45*feat[...,4])
    return Image.fromarray(rgb).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BICUBIC)

def draw_signal_channel(signal, lo, hi, width=2):
    signal=np.nan_to_num(np.asarray(signal,dtype=np.float32),nan=0,posinf=hi,neginf=lo)
    if hi<=lo: hi=lo+1e-6
    y=np.clip((signal-lo)/(hi-lo),0,1)
    xs=np.linspace(0,IMAGE_SIZE-1,len(y)); ys=(1-y)*(IMAGE_SIZE-1)
    canvas=Image.new("L",(IMAGE_SIZE,IMAGE_SIZE),0); draw=ImageDraw.Draw(canvas)
    pts=[(float(xs[i]),float(ys[i])) for i in range(len(y))]
    if len(pts)>1: draw.line(pts,fill=255,width=width)
    return np.asarray(canvas,dtype=np.uint8)

def gsr_v2_image(gsr_30s, scale):
    raw=np.asarray(gsr_30s,dtype=np.float32)
    tonic=smooth_gsr(raw); phasic=raw-tonic; deriv=np.gradient(raw).astype(np.float32)
    # R=raw, G=phasic, B=derivative: preserves complementary GSR information.
    r=draw_signal_channel(raw,scale["raw_lo"],scale["raw_hi"],3)
    g=draw_signal_channel(phasic,scale["phasic_lo"],scale["phasic_hi"],3)
    b=draw_signal_channel(deriv,scale["deriv_lo"],scale["deriv_hi"],2)
    return Image.fromarray(np.stack([r,g,b],axis=-1))

def build_segments(npz_file):
    data=np.load(npz_file,mmap_mode="r")
    labels=np.asarray(data["labels"]).astype(np.int32)
    phases=np.asarray([decode_phase(x) for x in data["phases"]])
    wis=np.asarray(data["window_indices"]).astype(np.int32)
    out=[]
    for phase in np.unique(phases):
        idx=np.where(phases==phase)[0]; idx=idx[np.argsort(wis[idx])]
        for start in range(0,len(idx)-SEGMENT_WINDOWS+1,SEGMENT_STRIDE):
            sel=idx[start:start+SEGMENT_WINDOWS]; w=wis[sel]
            if not np.all(np.diff(w)==1): continue
            labs=labels[sel]
            if not np.all(labs==labs[0]): continue
            out.append((phase,int(w[0]),sel,int(labs[0])))
    data.close(); return out

# Train-only GSR scale
train_subjects=set(base_manifest.loc[base_manifest.split=="train","subject"].astype(str))
raw_vals=[]; phasic_vals=[]; deriv_vals=[]
for npz in sorted(WINDOW_DIR.glob("*.npz")):
    if npz.stem not in train_subjects: continue
    d=np.load(npz,mmap_mode="r")
    for i in range(0,len(d["eda"]),10):
        x=np.asarray(d["eda"][i],dtype=np.float32); tonic=smooth_gsr(x)
        raw_vals.append(x); phasic_vals.append(x-tonic); deriv_vals.append(np.gradient(x))
    d.close()
raw_vals=np.concatenate(raw_vals); phasic_vals=np.concatenate(phasic_vals); deriv_vals=np.concatenate(deriv_vals)
GSR_SCALE={"raw_lo":float(np.percentile(raw_vals,1)),"raw_hi":float(np.percentile(raw_vals,99)),
           "phasic_lo":float(np.percentile(phasic_vals,1)),"phasic_hi":float(np.percentile(phasic_vals,99)),
           "deriv_lo":float(np.percentile(deriv_vals,1)),"deriv_hi":float(np.percentile(deriv_vals,99))}
del raw_vals,phasic_vals,deriv_vals; gc.collect()

records=[]
npz_files=sorted(WINDOW_DIR.glob("*.npz"))
for n,npz in enumerate(npz_files,1):
    subject=npz.stem; rows=base_manifest[base_manifest.subject.astype(str)==subject]
    if len(rows)==0: continue
    split=str(rows.split.iloc[0]); edirs=(EEG_DIR/subject); gdirs=(GSR_DIR/subject)
    edirs.mkdir(parents=True,exist_ok=True); gdirs.mkdir(parents=True,exist_ok=True)
    d=np.load(npz,mmap_mode="r"); eeg=d["eeg"]; eda=d["eda"]
    segs=build_segments(npz)
    for phase,start_w,sel,orig in segs:
        y=1 if orig==2 else 0; label="high" if y else "normal"; stem=f"{subject}_{phase}_s{start_w:03d}"
        ep=edirs/f"{stem}.png"; gp=gdirs/f"{stem}.png"
        if not (ep.exists() and gp.exists()):
            eeg30=np.concatenate([np.asarray(eeg[i],dtype=np.float32) for i in sel],axis=1)
            gsr30=np.concatenate([np.asarray(eda[i],dtype=np.float32) for i in sel],axis=0)
            eeg_v2_image(eeg30).save(ep,format="PNG",optimize=True)
            gsr_v2_image(gsr30,GSR_SCALE).save(gp,format="PNG",optimize=True)
        records.append({"subject":subject,"phase":phase,"start_window":start_w,"original_label_id":orig,
                        "binary_label_id":y,"binary_label":label,"split":split,"eeg_image":str(ep),"gsr_image":str(gp)})
    d.close(); gc.collect(); print(f"[{n:02d}/{len(npz_files)}] {subject}: {len(segs)} segments")

v2=pd.DataFrame(records); v2.to_csv(MANIFEST_OUT,index=False)
assert v2.groupby("subject").split.nunique().max()==1
assert v2.eeg_image.map(os.path.exists).all() and v2.gsr_image.map(os.path.exists).all()
print("\nV2 image dataset:",len(v2),"segments /",v2.subject.nunique(),"subjects")
print(v2.groupby(["split","binary_label"]).size())
print("NO SUBJECT LEAKAGE: OK")


In [ ]:
# ============================================================
# 3. BUILD BALANCED TF.DATA PIPELINES
# ============================================================
import tensorflow as tf
from sklearn.metrics import *

SEED=42
IMG_SIZE=224
BATCH_SIZE=32
MANIFEST=Path("/content/stress_image_dataset_binary_v2/image_manifest_binary_v2.csv")
MODEL_DIR=Path("/content/stress_binary_model_v2"); MODEL_DIR.mkdir(parents=True,exist_ok=True)

df=pd.read_csv(MANIFEST)
train_df=df[df.split=="train"].reset_index(drop=True)
val_df=df[df.split=="validation"].reset_index(drop=True)
test_df=df[df.split=="test"].reset_index(drop=True)
print("Train/Val/Test:",len(train_df),len(val_df),len(test_df))
print("Train classes:\n",train_df.binary_label.value_counts())

def read_png(path):
    x=tf.io.decode_png(tf.io.read_file(path),channels=3)
    return tf.cast(tf.image.resize(x,[IMG_SIZE,IMG_SIZE]),tf.float32)/255.0

def map_pair(e,g,y): return {"eeg_image":read_png(e),"gsr_image":read_png(g)},tf.cast(y,tf.float32)

def basic_ds(frame,shuffle=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.eeg_image.astype(str).values,frame.gsr_image.astype(str).values,frame.binary_label_id.astype(np.float32).values))
    if shuffle: ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(map_pair,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Balanced sampling is applied ONLY to the training split.
neg=train_df[train_df.binary_label_id==0]; pos=train_df[train_df.binary_label_id==1]
neg_ds=basic_ds(neg,True).repeat(); pos_ds=basic_ds(pos,True).repeat()
train_ds=tf.data.Dataset.sample_from_datasets([neg_ds,pos_ds],weights=[0.5,0.5],seed=SEED).take(max(len(neg),len(pos))*2//BATCH_SIZE+1)
val_ds=basic_ds(val_df); test_ds=basic_ds(test_df)
print("Balanced training batches:",tf.data.experimental.cardinality(train_ds).numpy())


In [ ]:
# ============================================================
# 4. V2 MODEL: SMALLER BRANCHES + SE + GATED FUSION
# ============================================================
# The previous model learned training-specific patterns (train acc ~88-89%
# vs validation ~62-63%). This model deliberately reduces capacity.

def se_block(x,ratio=8,name="se"):
    c=int(x.shape[-1]); s=tf.keras.layers.GlobalAveragePooling2D(name=name+"_gap")(x)
    s=tf.keras.layers.Dense(max(c//ratio,8),activation="swish",name=name+"_fc1")(s)
    s=tf.keras.layers.Dense(c,activation="sigmoid",name=name+"_fc2")(s)
    s=tf.keras.layers.Reshape((1,1,c))(s)
    return tf.keras.layers.Multiply(name=name+"_scale")([x,s])

def block(x,f,s,name):
    skip=x
    x=tf.keras.layers.SeparableConv2D(f,3,strides=s,padding="same",use_bias=False,name=name+"_c1")(x)
    x=tf.keras.layers.BatchNormalization(name=name+"_bn1")(x); x=tf.keras.layers.Activation("swish")(x)
    x=tf.keras.layers.SeparableConv2D(f,3,padding="same",use_bias=False,name=name+"_c2")(x)
    x=tf.keras.layers.BatchNormalization(name=name+"_bn2")(x); x=se_block(x,8,name+"_se")
    if s!=1 or int(skip.shape[-1])!=f:
        skip=tf.keras.layers.Conv2D(f,1,strides=s,padding="same",use_bias=False)(skip)
        skip=tf.keras.layers.BatchNormalization()(skip)
    return tf.keras.layers.Activation("swish")(tf.keras.layers.Add()([x,skip]))

def branch(inp,prefix):
    x=tf.keras.layers.Conv2D(24,5,strides=2,padding="same",use_bias=False,name=prefix+"_stem")(inp)
    x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.Activation("swish")(x)
    x=block(x,32,1,prefix+"_b1"); x=block(x,48,2,prefix+"_b2"); x=block(x,64,2,prefix+"_b3"); x=block(x,96,2,prefix+"_b4")
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.Dense(128,activation="swish")(x)
    x=tf.keras.layers.Dropout(0.35)(x)
    return x

eeg_in=tf.keras.Input((IMG_SIZE,IMG_SIZE,3),name="eeg_image")
gsr_in=tf.keras.Input((IMG_SIZE,IMG_SIZE,3),name="gsr_image")

# Conservative augmentation: no flips/large translations because image geometry carries signal meaning.
eeg_aug=tf.keras.Sequential([tf.keras.layers.RandomContrast(0.08),tf.keras.layers.GaussianNoise(0.015)],name="eeg_aug")
gsr_aug=tf.keras.Sequential([tf.keras.layers.RandomContrast(0.08),tf.keras.layers.GaussianNoise(0.015)],name="gsr_aug")

e=branch(eeg_aug(eeg_in),"eeg"); g=branch(gsr_aug(gsr_in),"gsr")
prod=tf.keras.layers.Multiply()([e,g]); diff=tf.keras.layers.Subtract()([e,g])
cat=tf.keras.layers.Concatenate(name="fusion_features")([e,g,prod,diff])
gate=tf.keras.layers.Dense(2,activation="softmax",name="modality_gate")(tf.keras.layers.Concatenate()([e,g]))
ew=tf.keras.layers.Multiply()([e,tf.keras.layers.Lambda(lambda z:z[:,0:1])(gate)])
gw=tf.keras.layers.Multiply()([g,tf.keras.layers.Lambda(lambda z:z[:,1:2])(gate)])
f=tf.keras.layers.Concatenate()([cat,ew,gw])
f=tf.keras.layers.Dense(192,activation="swish",kernel_regularizer=tf.keras.regularizers.l2(2e-4))(f)
f=tf.keras.layers.BatchNormalization()(f); f=tf.keras.layers.Dropout(0.45)(f)
f=tf.keras.layers.Dense(64,activation="swish",kernel_regularizer=tf.keras.regularizers.l2(2e-4))(f); f=tf.keras.layers.Dropout(0.30)(f)
out=tf.keras.layers.Dense(1,activation="sigmoid",name="stress_probability")(f)
model=tf.keras.Model([eeg_in,gsr_in],out,name="Stress_EEG_GSR_V2")
model.summary()

# Focal loss: emphasizes difficult examples without making the network larger.
def focal_bce(y_true,y_pred,gamma=1.5,alpha=0.5):
    y_true=tf.cast(y_true,tf.float32); y_pred=tf.clip_by_value(y_pred,1e-7,1-1e-7)
    pt=y_true*y_pred+(1-y_true)*(1-y_pred)
    a=y_true*alpha+(1-y_true)*(1-alpha)
    return tf.reduce_mean(-a*tf.pow(1-pt,gamma)*(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred)))

model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4,weight_decay=2e-4),loss=focal_bce,
              metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),tf.keras.metrics.AUC(name="auc")])


In [ ]:
# ============================================================
# 5. TRAIN 3-SEED ENSEMBLE
# ============================================================
# Multiple independently initialized compact models are averaged.
# Selection still uses VALIDATION only. The TEST split remains untouched.

EPOCHS=60
N_MODELS=3
all_hist=[]
val_probs=[]; test_probs=[]

for seed in [42,123,777][:N_MODELS]:
    print("\n"+"="*72); print("ENSEMBLE MODEL SEED",seed); print("="*72)
    tf.keras.utils.set_random_seed(seed)
    # Rebuild weights while preserving the exact architecture.
    fresh=tf.keras.models.clone_model(model)
    fresh.set_weights(model.get_weights())
    # Reinitialize deterministically by constructing a new model with seed through global RNG.
    # clone_model copies initial weights, so explicitly randomize by creating fresh via the functional graph.
    for layer in fresh.layers:
        if hasattr(layer,"kernel_initializer") and hasattr(layer,"kernel"):
            try: layer.kernel.assign(layer.kernel_initializer(tf.shape(layer.kernel)))
            except: pass
        if hasattr(layer,"bias_initializer") and hasattr(layer,"bias") and layer.bias is not None:
            try: layer.bias.assign(layer.bias_initializer(tf.shape(layer.bias)))
            except: pass
    fresh.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4,weight_decay=2e-4),loss=focal_bce,
                  metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"),tf.keras.metrics.AUC(name="auc")])
    path=MODEL_DIR/f"ensemble_seed_{seed}.weights.h5"
    cb=[tf.keras.callbacks.ModelCheckpoint(str(path),monitor="val_auc",mode="max",save_best_only=True,save_weights_only=True,verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_auc",mode="max",patience=10,restore_best_weights=True,verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc",mode="max",factor=0.5,patience=4,min_lr=2e-6,verbose=1)]
    h=fresh.fit(train_ds,validation_data=val_ds,epochs=EPOCHS,callbacks=cb,verbose=1)
    all_hist.append(h.history)
    val_probs.append(fresh.predict(val_ds,verbose=0).reshape(-1))
    test_probs.append(fresh.predict(test_ds,verbose=0).reshape(-1))
    fresh.save(MODEL_DIR/f"model_seed_{seed}.keras")
    print("Best validation AUC:",max(h.history["val_auc"]))

val_prob=np.mean(np.vstack(val_probs),axis=0)
test_prob=np.mean(np.vstack(test_probs),axis=0)
np.save(MODEL_DIR/"ensemble_validation_prob.npy",val_prob)
np.save(MODEL_DIR/"ensemble_test_prob.npy",test_prob)
print("Ensemble complete")


In [ ]:
# ============================================================
# 6. VALIDATION-ONLY THRESHOLD + STRICT TEST EVALUATION
# ============================================================

y_val=val_df.binary_label_id.values.astype(int)
y_test=test_df.binary_label_id.values.astype(int)

best_t=0.5; best_score=-1; best_macro=-1
for t in np.arange(0.20,0.81,0.01):
    p=(val_prob>=t).astype(int)
    bal=balanced_accuracy_score(y_val,p); mf=f1_score(y_val,p,average="macro",zero_division=0)
    if bal>best_score or (np.isclose(bal,best_score) and mf>best_macro): best_t=float(t); best_score=bal; best_macro=mf

pred=(test_prob>=best_t).astype(int)
acc=accuracy_score(y_test,pred); bal=balanced_accuracy_score(y_test,pred); macro=f1_score(y_test,pred,average="macro",zero_division=0)
auc=roc_auc_score(y_test,test_prob)
print("VALIDATION threshold:",best_t)
print("Validation balanced accuracy:",round(best_score,4),"macro F1:",round(best_macro,4))
print("\nSTRICT UNSEEN-SUBJECT TEST")
print("Accuracy          :",round(acc,4),f"({acc*100:.2f}%)")
print("Balanced Accuracy :",round(bal,4),f"({bal*100:.2f}%)")
print("Precision HIGH    :",round(precision_score(y_test,pred,zero_division=0),4))
print("Recall HIGH       :",round(recall_score(y_test,pred,zero_division=0),4))
print("F1 HIGH           :",round(f1_score(y_test,pred,zero_division=0),4))
print("Macro F1          :",round(macro,4))
print("ROC-AUC           :",round(auc,4))
print("\nClassification report:\n",classification_report(y_test,pred,target_names=["normal","high"],zero_division=0))
print("Confusion matrix:\n",confusion_matrix(y_test,pred,labels=[0,1]))

# Optional subject-level aggregation (reported separately, never used to claim segment accuracy).
test_out=test_df.copy(); test_out["prob"]=test_prob; test_out["pred"]=pred
subj_pred=[]; subj_true=[]
for s,g in test_out.groupby("subject"):
    subj_pred.append(int(g.prob.mean()>=best_t)); subj_true.append(int(g.binary_label_id.iloc[0]))
print("\nSUBJECT-LEVEL (mean probability) accuracy:",round(accuracy_score(subj_true,subj_pred)*100,2),"%")

# Save metrics
metrics={"target_accuracy":0.90,"segment_test_accuracy":float(acc),"balanced_accuracy":float(bal),"macro_f1":float(macro),"roc_auc":float(auc),"threshold":best_t,
         "subject_level_accuracy":float(accuracy_score(subj_true,subj_pred)),"strict_unseen_subject_test":True}
with open(MODEL_DIR/"v2_metrics.json","w") as f: json.dump(metrics,f,indent=2)
print("\n90% target achieved?", "YES" if acc>=0.90 else "NO — result is reported honestly; improve data/labels/features rather than falsifying test performance.")


In [ ]:
# ============================================================
# 7. CONFUSION MATRIX + TRAINING CURVES
# ============================================================
import matplotlib.pyplot as plt
cm=confusion_matrix(y_test,pred,labels=[0,1])
plt.figure(figsize=(6,5)); plt.imshow(cm); plt.xticks([0,1],["normal","high"]); plt.yticks([0,1],["normal","high"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("V2 Strict Test Confusion Matrix")
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.colorbar(); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
for i,h in enumerate(all_hist): plt.plot(h["val_auc"],label=f"Seed {[42,123,777][i]} val AUC")
plt.xlabel("Epoch"); plt.ylabel("Validation AUC"); plt.title("V2 Ensemble Validation AUC"); plt.legend(); plt.show()


In [ ]:
# ============================================================
# 8. SAVE CONFIG + PACKAGE RESULTS
# ============================================================
config={"model":"Stress_EEG_GSR_V2","task":"NORMAL vs HIGH stress","segment_seconds":30,"segment_stride_seconds":15,
        "eeg_bands":["0.5-4 Hz delta","4-8 Hz theta","8-13 Hz alpha","13-30 Hz beta","30-45 Hz gamma"],
        "gsr_channels":["raw","phasic","derivative"],"ensemble_seeds":[42,123,777][:N_MODELS],"threshold":best_t,
        "strict_subject_split":True,"target_accuracy":0.90}
with open(MODEL_DIR/"model_config_v2.json","w") as f: json.dump(config,f,indent=2)

import shutil
zip_path=shutil.make_archive("/content/stress_binary_model_v2_results","zip",root_dir=MODEL_DIR)
print("Results ZIP:",zip_path)
print("Keras weights are stored per ensemble seed in:",MODEL_DIR)
print("DONE")
